# 13.5 DPO如何把一對偏好變成更新方向？


沿用較佳−4、較差−3的策略log分數，固定參考也相同。先算相對差距：

`margin=(策略較佳−策略較差)−(參考較佳−參考較差)`。

本例margin為0。DPO用`L=−log(sigmoid(beta×margin))`作代價；beta是正的縮放係數。`sigmoid(z)=1/(1+exp(−z))`把差距轉到0與1之間，0變0.5，較大正值接近1。因此相對更偏向較佳答案，代價就降低。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.alignment import dpo_loss

chosen = torch.tensor([-4.0], requires_grad=True)
rejected = torch.tensor([-3.0], requires_grad=True)
loss = dpo_loss(chosen, rejected, torch.tensor([-4.0]), torch.tensor([-3.0]), beta=0.1)
loss.backward()
print("代價", round(loss.item(), 4))
print("較佳梯度", round(chosen.grad.item(), 4))
print("較差梯度", round(rejected.grad.item(), 4))

程式給的是四個log機率分數，不是裁判星等。beta0.1、margin0時，代價約0.6931，較佳分數梯度−0.05、較差+0.05。`backward()`只算敏感程度，沒有更新任何語言網路。

用學習率0.1做紙上下降，舊值減去學習率乘梯度：較佳變−3.995，較差變−3.005；差距從−1變−0.99，相對改善0.01。真模型則透過共同權重改變每步機率，不能獨立隨意改每篇分數。

候選排序與實際回答還要分開測。保存實驗中，`4+2=?`的正確候選6比錯候選7有更高整段機率，模型自由生成卻回答8。它並未被限制只能選這兩篇，因此「這一對排對」仍不是「回答算對」。

練習把策略較佳分數改−5，參考保持不變。margin變−1，代價上升，較佳梯度仍為負，下降方向仍是提高相對偏好。

<details>
<summary>補充：來源、完整設定與既有實測紀錄</summary>

真正訓練後，候選比較與自由生成也要各測一次。beta 0.1版本對最後檢查題`4+2=?`，完整正確候選`6`的log分數約−24.88083，比錯誤候選`7`的−28.11597高；但是讓它自行選字時，實際回答是`8`，接著正常EOS結束。模型沒有被限制只能二選一，候選排序也不是自由生成答案的驗證。這一題可以同時記為「6比7更受偏好」及「實際加法答錯」，兩項都是真的。

</details>
